# Machine Learning
Pasar de serie temporal a problema supervisado, variables retardadas, predictores ambientales y validación temporal.

1. Problema científico: los métodos de Machine Learning permiten aprender relaciones estadísticas entre una variable de interés y un conjunto de variables explicativas a partir de observaciones históricas. A diferencia de los modelos estadísticos clásicos de series temporales, estos métodos no necesitan definir explícitamente una estructura de tendencia, estacionalidad o dependencia temporal, aunque pueden incorporar información temporal entre sus variables de entrada.

2. Datos necesarios: la división entre entrenamiento y evaluación debe tener en cuenta la estructura temporal de los datos.

3. Fundamento:

## Ejemplo reproducible: ¿Cómo se aplica a datos reales o a un caso didáctico?

En este notebook se utiliza Random Forest (RF), un algoritmo de aprendizaje supervisado, para explorar la predicción de una variable oceanográfica a partir de otras variables ambientales. Random Forest no es, por sí mismo, un modelo de series temporales. Es un algoritmo de aprendizaje supervisado que puede utilizarse para predecir una variable oceanográfica a partir de otras variables, incluidas las observaciones pasadas, si se diseña la matriz de features de esa manera.

## Ejemplo propuesto: Random Forest (RF)

**Objetivo** predecir clorofila (fluorescencia) futura a partir de información disponible en el momento de la predicción, como medidas pasadas, estacionalidad y variables ambientales conocidas o pronosticadas.

**Datos:** Fluorescencia superficie mensual (E2GI) + temperatura superficie mensual (E2GI).

**Ventajas** Los métodos de ML encuentran patrones complejos e interacciones entre muchas variables.

**Desventajas** Es una "caja negra"; no respeta las leyes de la física por sí solo. Las relaciones identificadas por el modelo no deben interpretarse automáticamente como mecanismos físicos o biológicos, ni como relaciones causales.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from skforecast.recursive import ForecasterRecursive

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
import shap
from skforecast.recursive import ForecasterRecursive

## 1 · Datos
Leer datos hidrografía, no tenemos frecuencia diaria.

In [7]:
df = pd.read_csv(
    "../data/processed/Hidro_E2GI_surface_monthly.csv",
    index_col="mes")

# Conservar el calendario mensual completo
df.index = pd.to_datetime(df.index)
df = df.sort_index()
df = df.asfreq("MS")

# Seleccionar fluorescencia y temperatura SIN eliminar meses
df_model = df[["fluor", "temp"]].copy()

print("Frecuencia:", df_model.index.freq)
print("Observaciones del calendario:", len(df_model))
print("Valores ausentes:\n", df_model.isna().sum())
print(df_model.describe().round(2))

Frecuencia: <MonthBegin>
Observaciones del calendario: 256
Valores ausentes:
 fluor    46
temp     46
dtype: int64
        fluor    temp
count  210.00  210.00
mean     1.33   16.05
std      1.49    3.11
min      0.00   11.35
25%      0.29   13.15
50%      0.97   15.32
75%      1.73   18.95
max     11.44   23.40


## 2 · Matriz de features
Construir la matriz de features para la fluorescencia (lags + rolling + calendario + Fourier + temperatura retardada).
Función reutilizable. Importante: rolling siempre center=False y aplicado en shift(1) para que no haya fuga del futuro.

In [8]:
def construir_features(df, target="fluor", exog="temp"):
    """Devuelve X (DataFrame) e y (Series) listos para fit/predict."""
    out = pd.DataFrame(index=df.index)

    # 1) Lags de la propia fluorescencia
    for lag in (1, 6, 12):
        out[f"q_lag{lag}"] = df[target].shift(lag)

    # 2) Rolling de la temperatura (siempre con shift previo)
    for w in (1, 2, 3):
        out[f"q_mean{w}"] = df[target].shift(1).rolling(w).mean()
        out[f"q_max{w}"] = df[target].shift(1).rolling(w).max()
        out[f"q_std{w}"] = df[target].shift(1).rolling(w).std()

    # 3) Temperarura retardada y ¿acumulada?
    for lag in (1, 2):
        out[f"p_lag{lag}"] = df[exog].shift(lag)
    for w in (1, 2):
        out[f"p_acum{w}d"] = df[exog].rolling(w).sum().shift(1)

    # 4) Calendario + Fourier
    idx = out.index
    out["mes"] = idx.month
    out["sin_an"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    out["cos_an"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)

    # 5) Target
    out["y"] = df[target]

    # dropna() pierde el atributo freq → lo reconstruimos con asfreq('D')
    return out.dropna().asfreq("MS")


feat = construir_features(df_model)
print(f"Features: {feat.shape[1] - 1}   Filas válidas: {len(feat):,}   Freq: {feat.index.freq}")
feat.tail(3).round(2)

Features: 19   Filas válidas: 0   Freq: <MonthBegin>


,q_lag1,q_lag6,q_lag12,q_mean1,q_max1,q_std1,q_mean2,q_max2,q_std2,q_mean3,q_max3,q_std3,p_lag1,p_lag2,p_acum1d,p_acum2d,mes,sin_an,cos_an,y
mes,,,,,,,,,,,,,,,,,,,,


## 3 · Split temporal
Los datos suelen dividirse en un conjunto de entrenamiento (train) y otro de prueba (test). El modelo se ajusta con el primero y sus predicciones se comparan con los valores observados en el segundo. Cuando existen observaciones temporal o espacialmente relacionadas, no se barajan.
Train hasta 2017, test 2018-2020.

In [ ]:
split = pd.Timestamp(\"2018-01-01\")\n",
    "train = feat.loc[:split].iloc[:-1]\n",
    "test = feat.loc[split:]\n"
                     y_train, y_test = train[\"y\"], test[\"y\"]\n",
    "X_train = train.drop(columns=\"y\")\n",
    "X_test = test.drop(columns=\"y\")\n",
    "print(f\"Train: {len(train):,} obs   Test: {len(test):,} obs   Features: {X_train.shape[1]}\")"


## 4 · Baseline
Tres que un modelo serio debe batir:
skforecast
ForecasterRecursive

In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(((y - yhat) ** 2).mean()))


def mae(y, yhat):
    return float((y - yhat).abs().mean())


# Baseline 1: persistencia (yhat = y_{t-1})
pred_pers = X_test["q_lag1"]

# Baseline 2: persistencia anual (yhat = y_{t-365})
pred_pers_anual = X_test["q_lag365"]

# Baseline 3: climatología (media histórica del día del año)
clim = y_train.groupby(y_train.index.dayofyear).mean()
pred_clim = y_test.index.dayofyear.map(clim)

for nombre, pred in [
    ("Persistencia (lag-1)", pred_pers),
    ("Persistencia anual (lag-365)", pred_pers_anual),
    ("Climatología (doy)", pred_clim),
]:
    print(f"{nombre:35s} RMSE={rmse(y_test, pred):.3f}  MAE={mae(y_test, pred):.3f}")

In [ ]:
exog_cols = [c for c in X_train.columns if not c.startswith("q_")]
y_train_s = pd.Series(y_train.values, index=y_train.index, name="caudal")
y_test_s = pd.Series(y_test.values, index=y_test.index, name="caudal")
exog_train_df = X_train[exog_cols]
exog_test_df = X_test[exog_cols]

forecaster = ForecasterRecursive(
    regressor=Ridge(alpha=1.0),
    lags=[1, 2, 3, 7, 14, 30, 90, 365],
)
forecaster.fit(y=y_train_s, exog=exog_train_df)

# Predicción 1 día (rolling): usamos el último valor real disponible
pred_ridge = forecaster.predict(steps=len(y_test_s), exog=exog_test_df)

print(
    f"Ridge skforecast (recursivo h={len(y_test_s)})  "
    f"RMSE={rmse(y_test, pred_ridge):.3f}  MAE={mae(y_test, pred_ridge):.3f}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, color="black", lw=1, label="observado")
ax.plot(y_test.index, pred_pers.values, color="#999", lw=0.8, ls=":", label="persistencia")
ax.plot(y_test.index, pred_ridge.values, color="#c2410c", lw=1, ls="--", label="Ridge skforecast")
ax.set_ylabel("Q (m³/s)")
ax.legend()
ax.set_title("Test 2018-2020 — predicción 1 día (con feedback)")
plt.tight_layout()

## 5 · RF


Ajustar Random Forest y XGBoost vía skforecast con la misma matriz de features que el baseline. Reflexión: ¿en qué casos gana el ML clásico vs SARIMAX/Prophet de Sesión 2?

In [ ]:
LAGS = [1, 2, 3, 7, 14, 30, 90, 365]

modelos = {
    "Ridge": Ridge(alpha=1.0),
    "RF": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1, verbosity=0, random_state=0
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=-1, random_state=0, verbose=-1
    ),
}

preds = {}
for nombre, regr in modelos.items():
    f = ForecasterRecursive(regressor=regr, lags=LAGS)
    f.fit(y=y_train, exog=X_train)
    preds[nombre] = f.predict(steps=len(y_test), exog=X_test)
    print(f"{nombre:8s} entrenado")

## 6 · Métricas
Entre las métricas de regresión más habituales se encuentran:

MAE (Mean Absolute Error): error absoluto medio, expresado en las unidades de la variable objetivo.

RMSE (Root Mean Squared Error): raíz del error cuadrático medio, que penaliza especialmente los errores grandes.

R² (coeficiente de determinación): compara el error del modelo con la variabilidad de los valores observados respecto a su media.

In [ ]:
def rmse(y, yhat):
    return float(np.sqrt(((y - yhat) ** 2).mean()))


def mae(y, yhat):
    return float((y - yhat).abs().mean())


tabla = pd.DataFrame(
    {nombre: {"RMSE": rmse(y_test, p), "MAE": mae(y_test, p)} for nombre, p in preds.items()}
).T.round(3)
print(tabla)

## 7 · Visualización
plot *clorofila superficie*, diferenciando periodo de entrenamiento y periodo de test.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_test.index, y_test.values, color="black", lw=1, label="observado")
colores = {"Ridge": "#999", "RF": "#2563eb", "XGBoost": "#c2410c", "LightGBM": "#16a34a"}
for n, p in preds.items():
    ax.plot(p.index, p.values, color=colores[n], lw=0.9, ls="--", label=n, alpha=0.9)
ax.set_ylabel("Q (m³/s)")
ax.legend(ncol=5)
ax.set_title("Predicción recursiva test 2018-2020")
plt.tight_layout()

4. Validación: ¿Cómo sabemos si predice razonablemente bien?

La evaluación debe determinar si el modelo predice adecuadamente observaciones que no ha utilizado durante su entrenamiento. Un buen ajuste sobre los datos de entrenamiento no garantiza que el modelo generalice a nuevos datos. Si se pretende estimar el rendimiento en condiciones ambientales nuevas, debe comprobarse si esas condiciones están representadas en el entrenamiento (dataset shift).

5. Limitaciones: ¿Cuándo falla y qué conclusiones no permite extraer?
ML es útil para identificar relaciones predictivas complejas entre variables oceanográficas, pero presenta limitaciones que deben considerarse al interpretar sus resultados.

El modelo puede tener dificultades cuando se aplica a condiciones ambientales fuera del rango representado en los datos de entrenamiento. Del mismo modeo, un muestreo sesgado, escaso o poco representativo puede limitar la capacidad de generalización.

Aunque las importancias de variables y otras herramientas de interpretación permiten examinar el comportamiento del modelo, no proporcionan por sí mismas una explicación mecanística del proceso.

El modelo puede no reproducir adecuadamente episodios raros o valores fuera del rango observado durante el entrenamiento (eventos extremos). MAE, RMSE, MAPE, sMAPE: ventajas y trampas.
Evaluación específica de eventos extremos?
Métricas y evaluación: Compara MAE, RMSE y, si resulta útil, \(R^2\). Analiza qué modelo predice mejor los cambios de concentración y los episodios de valores elevados.

6. Información adicional ¿Dónde puede profundizar quien quiera utilizarlo en investigación?

https://scikit-learn.org/stable/modules/ensemble.html

## A mayores: Redes neuronales para series temporales
Las redes neuronales artificiales constituyen otra familia de métodos de aprendizaje automático que puede aplicarse a la predicción de variables oceanográficas. Por lo que yo se, existen arquitecturas diseñadas para trabajar con datos secuenciales y series temporales, como las redes recurrentes (RNN), las redes LSTM (Long Short-Term Memory), las redes convolucionales temporales y los Transformers.

##### **Atención** Este notebook no incluye estas técnicas, pero las dejo aquí mencionadas porque tenemos compañeros que han utilizado NN ampliamente en tareas de clasificación de imágenes. Para profundizar en esta parte, la persona de contacto en el COG sería Ángel López-Urrutia.